# Week 1 Day 2 Lab-Introducing UnderCover

In [2]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field


## Connect to OpenRouter

In [3]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'

if openrouter_api_key:
    print('OpenRouter API key loaded successfully.')
else: 
    print('No OpenRouter API key found.')

openrouter = OpenAI(api_key=openrouter_api_key,
                        base_url=OPENROUTER_BASE_URL,
                        timeout=120,
                        max_retries=5)

OpenRouter API key loaded successfully.


In [4]:
messages = [
    {
        "role":"system",
        "content":"You are a helpful assistant who answers in the style of a pirate"
    },
    {
        "role": "user",
        "content": "Ahoy, matey! Tell me a pirate joke."
    }
]

response = openrouter.chat.completions.create(
            model="openai/gpt-6-luna",
            messages=messages
        )
display(Markdown(response.choices[0].message.content))


Why did the pirate go to the gym? To improve his **arrrrms**, matey! 🏴‍☠️

## Part 1: Load And Examine The Data

In [5]:
from pathlib import Path
print(Path.cwd())

/Users/michaelperrine/Udemy/Donner/undercover/MyWork/UndercoverProject/week1


In [6]:
DATA = Path("../../../data")

def load(name):
    """ Read one of the JSON seed files"""
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def get_sample_data(tiny=False, held_out=False):
    """ Return a list of claims, and the handlers' decisions on them keyed by claim id.
    
    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision['claim_id'] for decision in decisions if decision['tiny']}
        claims = [claim for claim in claims if claim['claim_id'] in chosen]
        decisions = [decision for decision in decisions if decision['tiny']]
    labels = {decision['claim_id']: decision for decision in decisions}
    return claims, labels

In [7]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy['policy_id']: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding='utf-8')

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

25 claims, 25 labels, 50 policies
The policy wording is 1,369 words


In [8]:
claims[0]

{'claim_id': 'CLM-1001',
 'policy_id': 'POL-2001',
 'customer_id': 'CUS-3001',
 'claim_type': 'trip_delay',
 'submitted_on': '2026-06-24',
 'incident_date': '2026-06-18',
 'incident_location': 'Rome, Italy',
 'summary': 'Flight home canceled by the airline, rebooked the next morning',
 'story': 'My flight home from Rome on June 18 was canceled by Meridian Air about two hours before departure because of a crew problem. The next seat they could give me was at 6:55 the following morning. I booked a hotel near the airport, had dinner and breakfast there, and took taxis between the airport and the hotel. I am claiming the hotel, the meals and the taxis, and all of the receipts are attached.',
 'items': [{'description': 'Airport hotel, one night',
   'category': 'accommodation',
   'amount': 245.0},
  {'description': 'Dinner and breakfast', 'category': 'meals', 'amount': 95.0},
  {'description': 'Taxis between the airport and the hotel',
   'category': 'transport',
   'amount': 80.0}],
 'amo

In [9]:
labels[claims[0]['claim_id']]

{'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'over_approval_limit': False,
 'handler_id': 'HND-02',
 'decided_on': '2026-06-26',
 'reason': 'The carrier canceled the flight for a crew problem and the delay was 19 hours, well past the 6 hour waiting period on Premium. The hotel, meals and taxis are reasonable and all have receipts. The total is inside the $800 maximum.',
 'reference': 'Policy wording, section 3',
 'calculation': [{'step': 'Eligible expenses with receipts', 'amount': 420.0}],
 'tests': 'A routine trip delay with complete documents',
 'difficulty': 'routine',
 'failure_class': '',
 'needs': [],
 'tone': 'plain',
 'tiny': False}

## Explore the data

In [11]:
rows = []
for claim in claims:
    label = labels[claim['claim_id']]
    rows.append({
        'claim_id': claim['claim_id'],
        'summary': claim['summary'],
        'claim_type': claim['claim_type'],
        'amount_claimed': claim['amount_claimed'],
        'difficulty': label['difficulty'],
        'decision': label['decision'],
        'payout': label['payout']
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,claim_type,amount_claimed,difficulty,decision,payout
0,CLM-1001,"Flight home canceled by the airline, rebooked ...",trip_delay,420.0,routine,approve,420.0
1,CLM-1002,Suitcase lost on a connecting flight,baggage,1150.0,routine,approve,580.0
2,CLM-1003,"Hospital visit abroad after a fall, fractured ...",emergency_medical,3800.0,routine,approve,3750.0
3,CLM-1004,Checked bag arrived 31 hours late,baggage_delay,186.0,routine,approve,186.0
4,CLM-1005,Passport eaten by a giraffe at breakfast,passport,243.0,routine,approve,243.0
5,CLM-1006,Hit by a falling coconut while reading under a...,emergency_medical,425.0,routine,approve,425.0
6,CLM-1007,Phone dropped in the hotel shower,baggage,1099.0,routine,approve,50.0
7,CLM-1008,Trip to Seattle canceled after a tornado carri...,trip_cancellation,2440.0,routine,approve,2240.0
8,CLM-1009,Camera snatched on the Barcelona metro,baggage,980.0,routine,approve,930.0
9,CLM-1010,Emergency dental treatment in Las Vegas for a ...,emergency_medical,1350.0,routine,decline,0.0


In [12]:
home_alone = [claim for claim in claims if claim['claim_id'] == 'CLM-1012'][0]
decision = labels['CLM-1015']

display(Markdown(f"""**{home_alone['summary']} 
{home_alone['story']} **Handler:** {decision['decision']}
${decision['payout']:,.2F}. {decision['reason']}"""))

**Valise stolen from a train at the Gare de Lyon 
On July 2 I took the train from Paris to Lausanne to join my husband, who was working there. At the Gare de Lyon I put my valise on the luggage rack in our carriage and stepped off the train to buy a bottle of water for the journey. I was gone for perhaps ten minutes. When I came back the valise was gone. It held my clothes and, far worse, all of my husband's writing: the printed manuscript of his novel, his notebooks and the backup drive, which I had packed so that he could work on everything in Switzerland. There are no other copies. I reported it to the police at the station that afternoon. I have valued the manuscript and notebooks at $1,900, though my husband would call that figure an insult. **Handler:** decline
$0.00. Trip Interruption pays only for the reasons listed in the wording. Returning home because a child was left behind is not one of them, and nobody was sick or injured.

In [13]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim['claim_id'] == 'CLM-1010'][0]
decision = queue_labels['CLM-1010']

display(Markdown(f"""**{hangover['summary']}**
{hangover['story']} **Handler:** {decision['decision']}
${decision['payout']:,.2f}. {decision['reason']} """))

**Emergency dental treatment in Las Vegas for a missing front tooth**
I was in Las Vegas from June 19 for a friend's bachelor party. On the morning of June 21 I woke up on the floor of our suite missing an upper front tooth. I have no memory of the night before, and neither do my friends. I am a dentist, and I have to say that the extraction is consistent with my own technique. I went to an emergency dental clinic that morning for an exam, an X-ray and a temporary bridge, and their bill and notes are attached. There was also a tiger in the bathroom of the suite, which I mention only for completeness. I am not claiming for anything connected with the tiger. **Handler:** decline
$0.00. The treatment was needed, and the bill and the notes are in order. The general exclusions rule out losses while the customer is impaired by alcohol or drugs. The clinic's notes record heavy drinking through the evening, no memory of the night and an extraction that the customer believes he carried out himself during it, so the injury happened while he was impaired and no benefit pays. 

In [14]:
fig = px.histogram(data, 
                   x='claim_type',
                   color='decision',
                   title='Claims by type')
fig.show()

In [15]:
fig = px.histogram(data,
                   x='difficulty',
                   color='decision',
                   category_orders={'difficulty': ['routine', 'hard', 'manual']},
                   title='Claims by difficulty')
fig.show()

## Part 2: Running With Some LLMs

In [16]:
class ClaimDecision(BaseModel):
    """ A claims handler's decision on one claim."""

    rationale: str = Field(description='2-3 sentences to explain the decision along with the relevant section of the policy')
    decision: Literal['approve', 'decline', 'refer']
    payout: float = Field(description='The amount to pay  in US dollars, or 0 for a decline or a referral')

### The prompt

In [17]:
INSTRUCTIONS = """ 
You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS} {policy_wording}"""

def make_user_prompt(claim):
    """ The customer's policy record ant the claim, as JSON."""
    policy = policies[claim['policy_id']]
    return f"""# Policy record

{json.dumps(policy, indent=2)} 

# Claim

{json.dumps(claim, indent=2)}
 """

def make_messages(claim):
    """ Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return[
        {'role': 'system',
         'content': SYSTEM_PROMPT
         },
         {
             'role': 'user',
             'content': make_user_prompt(claim)
         }
    ]

### Calling the LLM

In [18]:
def decide(claim, model, effort):

    # Ask one model to decide one claim.
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
        extra_body={'provider': {'require_parameters': True}},
    )
    answer = response.choices[0].message.parsed
    return {
        'model': model,
        'effort': effort,
        'technique': 'LLM',
        'claim_id': claim['claim_id'],
        'decision': answer.decision,
        'payout': answer.payout,
        'rationale': answer.rationale,
        'cost': response.usage.cost,
        'seconds': round(time.time() - start, 1)

    }

In [19]:
first = decide(claims[0], 'openai/gpt-6-luna', 'low')
first

{'model': 'openai/gpt-6-luna',
 'effort': 'low',
 'technique': 'LLM',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': 'Approve under Section 3: Meridian Air confirms the return flight was canceled because of a crew problem, and the delay from the scheduled June 18 departure until the June 19 departure exceeds the Premium plan’s 6-hour waiting period. The attached receipts support $420 in reasonable accommodation, meal and local transport expenses; this is within the $800 Trip Delay maximum, with no deductible applying.',
 'cost': 0.0003948,
 'seconds': 1.9}

### Run Multiple Models

In [20]:
def is_right(answer, label):
    # True when the decision matches the handler's and the payout is within a dollar.
    return answer['decision'] == label['decision'] and abs(answer['payout'] - label['payout']) <= 1

def run(model, effort, claims, labels):
    # Run the model on the claims and compare against the labels.
    results = []
    for claim in claims:
        answer = decide(claim, model,effort)
        label = labels[claim['claim_id']]
        results.append({**answer, 
                        'difficulty': label['difficulty'],
                        'right': is_right(answer, label)})
    right = sum(result['right'] for result in results)
    cost = sum(result['cost'] for result in results)
    seconds = sum(result['seconds'] for result in results)
    print(f"{model} at {effort}: {right} of {len(results)} right for ${cost:.4f} in {seconds:.0f} seconds")
    return results

In [21]:
luna_low = run('openai/gpt-6-luna', 'low',claims, labels)

openai/gpt-6-luna at low: 18 of 25 right for $0.0045 in 54 seconds


In [22]:
luna_medium = run('openai/gpt-6-luna', 'medium', claims, labels)

openai/gpt-6-luna at medium: 18 of 25 right for $0.0062 in 100 seconds


In [23]:
luna_high = run('openai/gpt-6-luna', 'high', claims, labels)

openai/gpt-6-luna at high: 19 of 25 right for $0.0071 in 110 seconds


In [27]:
deepseek = run('deepseek/deepseek-v4.1-flash','high', claims, labels)

deepseek/deepseek-v4.1-flash at high: 19 of 25 right for $0.0407 in 172 seconds


In [25]:
gemini = run('google/gemini-3.5-flash-lite', 'low', claims, labels)

google/gemini-3.5-flash-lite at low: 14 of 25 right for $0.0326 in 31 seconds


In [33]:
import time

start = time.time()

response = openrouter.with_options(
    timeout=60.0,
    max_retries=0,
).chat.completions.parse(
    model="qwen/qwen3.5-397b-a17b",
    messages=make_messages(claims[0]),
    response_format=ClaimDecision,
    extra_body={"reasoning": {"effort": "low"}},
    max_tokens=4096,
)

print(f"Completed in {time.time() - start:.1f} seconds")
print(response.choices[0].message.parsed)

Completed in 11.4 seconds
rationale="The claim is for Trip Delay under the Premium plan. The delay was caused by a crew problem, which is a covered reason. The delay duration (approx. 19 hours) exceeds the Premium plan's 6-hour waiting period. The claimed expenses (accommodation, meals, transport) are eligible reasonable extra costs incurred during the delay. All required documents (carrier confirmation and receipts) are attached. The total claimed amount ($420) is within the Premium plan's Trip Delay maximum of $800. No deductible applies to Trip Delay benefits." decision='approve' payout=420.0


In [34]:
qwen = run('qwen/qwen3.5-397b-a17b', 'high', claims, labels)

qwen/qwen3.5-397b-a17b at high: 18 of 25 right for $0.3966 in 411 seconds


# Results by Difficulty

In [35]:
scored = pd.DataFrame(luna_low + luna_medium + luna_high + deepseek + gemini + qwen)
scored.pivot_table(index=['model', 'effort'], 
                   columns='difficulty',
                   values='right',
                   aggfunc='sum',
                   sort=False)

difficulty                           routine  hard  manual
model                        effort                       
openai/gpt-6-luna            low          14     3       1
                             medium       14     4       0
                             high         14     5       0
deepseek/deepseek-v4.1-flash high         14     5       0
google/gemini-3.5-flash-lite low          14     0       0
qwen/qwen3.5-397b-a17b       high         15     3       0

### Where A Model Got It Wrong

In [36]:
def show_misses(results, labels):
    """ Show each claim in a set of results that the model got wrong,
    with its rationale and the handler's reason.
    """
    for result in results:
        if not result['right']:
            label = labels[result['claim_id']]
            display(Markdown(f"""**{result['claim_id']},
            {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f},
            where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))

show_misses(luna_low, labels)


    

**CLM-1002,
            routine**: refer $0.00,
            where the handler said approve $580.00

- Model: The loss occurred during the covered trip, was reported to the carrier, and the carrier’s report and settlement letter are attached. However, receipts are missing for the shirts and toiletry kit, and section 11 requires the documents named for the benefit; the claim therefore needs to wait for the missing supporting documents. Under sections 5 and 10, the $400 carrier payment would also be deducted, with the Premium plan’s $300 per-item limit and $50 deductible applying to any eligible payout.
- Handler: The loss was reported to the carrier and the carrier has declared the bag lost. The suit is over the $300 per-item limit on Premium, so it counts at $300. The carrier's $400 comes off, then the $50 deductible.
- Tests: Payout arithmetic: the per-item limit, the carrier's payment and the deductible in order


**CLM-1019,
            manual**: approve $210.00,
            where the handler said refer $0.00

- Model: The carrier confirms a nine-hour delay caused by a crew problem, exceeding the Premium plan’s six-hour waiting period, and the required receipts and carrier confirmation are attached. The claimed $210 in accommodation, meals and local transport is eligible under Section 3 and is below the $800 Trip Delay maximum; no deductible applies under Section 10.
- Handler: On its own the claim is payable: a 9 hour delay on Premium with receipts. It is the customer's third claim in 12 months, after claims filed in October 2025 and March 2026, so it goes to Special Investigations for review before any payment.
- Tests: A rule found only in the claims manual: the third claim in 12 months. It needs the customer's claim history


**CLM-1020,
            manual**: refer $0.00,
            where the handler said approve $240.00

- Model: The loss occurred during the covered trip and the replacement receipt supports the claimed $240 expense, but section 5 requires theft to be reported to local police within 24 hours and a copy of the police report. The customer says the police did not take a report and has supplied only a tour guide’s letter, so a required document is missing; the claim should wait for review or further documentation.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1022,
            manual**: refer $0.00,
            where the handler said approve $155.00

- Model: The carrier confirms a 30-hour, 15-minute delay caused by severe weather, which meets the First plan’s 3-hour waiting period under section 3. Section 3 requires receipts for claimed extra costs; receipts are provided for the $85 rebooking fee, but not for the $140 meals, so the claim needs review or further documentation before payment can be decided.
- Handler: Severe weather held the train for 30 hours 15 minutes, past the 3 hour waiting period on First, and the railway confirms the times and the cause. The rebooking fee has a receipt. The meals have none, so the manual's flat allowance applies: $35 for one traveler for each 24 hours of delay or part of 24 hours, and 30 hours is two days. The incident on board does not change the cause of the delay.
- Tests: A rule found only in the claims manual: the meal allowance without receipts, over a delay of more than one day


**CLM-1023,
            hard**: approve $61.00,
            where the handler said refer $0.00

- Model: The delay exceeded the First plan’s 3-hour waiting period, and the story identifies a mechanical fault, a covered cause under Section 3. The carrier certificate and meal receipts are attached; the $61 in reasonable meal expenses is within the $1,500 Trip Delay maximum, with no deductible under Section 10.
- Handler: The delay of 9 hours 17 minutes is well past the 3 hour waiting period on First and the meal receipts are in order. Trip Delay pays only for the causes the wording lists, and it asks for the carrier's confirmation of the delay and its cause. The certificate confirms the times and gives no cause, so the fuel pump fault rests on the customer's account alone. Document request: Northwind's confirmation of the reason for the delay. The claim appears payable at $61 once it arrives.
- Tests: A required document that is present and incomplete: the carrier confirms the delay and leaves out the cause


**CLM-1024,
            manual**: approve $300.00,
            where the handler said refer $0.00

- Model: The phone was stolen during the covered trip and reported to local police the same day, with the required police report and purchase receipt attached. Under Section 5, theft is covered, but the Premium plan’s $300 per-item limit applies; there is no Electronics Plus add-on, and no deductible applies to baggage claims under Section 1. The claim was filed within 90 days of the incident, so the payable amount is $300 under Section 10.
- Handler: On its own the claim is payable at $250: a theft from the person, reported the same day, with the phone at the $300 limit less the $50 deductible. The customer's history shows claim CLM-0987 for the same theft on the same date, filed on July 16, approved and paid at $250. The same loss claimed twice goes to Special Investigations, with no comment to the customer beyond saying the claim is being reviewed. It may well be an honest mistake, since she says she was not sure the first claim went through.
- Tests: The customer's claim history: the same loss already claimed and paid. Nothing in the claim itself shows it


**CLM-1025,
            hard**: approve $71.00,
            where the handler said refer $0.00

- Model: The carrier confirmation and receipts document a 14-hour delay and $71 in meal expenses. A crew-hours problem qualifies under Section 3, and the delay exceeds Economy’s 12-hour waiting period; because this occurred on the way home, cover continues until the traveler gets home. Meals are eligible reasonable extra costs, and no deductible applies to Trip Delay.
- Handler: The delay of 14 hours 15 minutes is past the 12 hour waiting period on Economy and the meal receipts are in order. Trip Delay pays only for the causes the wording lists, and it asks for the carrier's confirmation of the delay and its cause. The certificate confirms the times and gives no cause, so the crew problem rests on the customer's account alone. Document request: Skylark's confirmation of the reason for the delay. The claim appears payable at $71 once it arrives.
- Tests: A required document that is present and incomplete: the carrier confirms the delay and leaves out the cause


# What It Cost

In [37]:
scored.groupby(['model', 'effort'], sort=False)[['right', 
                                                 'cost', 
                                                 'seconds']].sum()

right      cost  seconds
model                        effort                          
openai/gpt-6-luna            low        18  0.004517     54.0
                             medium     18  0.006163    100.3
                             high       19  0.007126    110.3
deepseek/deepseek-v4.1-flash high       19  0.040744    171.9
google/gemini-3.5-flash-lite low        14  0.032631     30.6
qwen/qwen3.5-397b-a17b       high       18  0.396586    410.6

## Part 3: Optional Extras

### Sol at High

In [38]:
sol = run('openai/gpt-6-sol', 'high', claims, labels)

openai/gpt-6-sol at high: 19 of 25 right for $0.1259 in 103 seconds


### Jev

In [40]:
JEV = 'typesafe/jev-1.13'
JEV_QUESTION = {
    'type': 'choice',
    'instructions': "You are a claims handler at UnderCover Travel Insurance. Decide `claim` using `policy_wording` and the customer's `policy` record.",
    'criteria': {
        'approve': 'The wording covers the claim and the required documents are attached.',
        'decline': 'The wording does not cover the claim .',
        'refer': 'A colleague needs to look at it, because a required document is missing or the story and the documents disagree, or the wording does not settle the question.',

    },
}
openrouter_decisions = OpenRouter(api_key=openrouter_api_key, timeout_ms=60_000)

def decide_with_jev(claim):
    """ Ask Jev for the decision on one claim, with its confidence """
    state = {'policy_wording': policy_wording, 
             'policy': policies[claim['policy_id']],
             'claim': claim}
    start = time.time()
    response = openrouter_decisions.alpha.decisions.create(
        model=JEV,
        state=state,
        questions={'decision': JEV_QUESTION})
    answer = response.answers['decision']
    return{
        'claim_id': claim['claim_id'],
        'decision': answer.choice,
        'confidence': answer.confidence,
        'right': answer.choice == labels[claim['claim_id']]['decision'],
        'cost': response.usage.cost,
        'seconds': round(time.time() - start, 2)
    }

In [41]:
jev = pd.DataFrame([decide_with_jev(claim) for claim in claims])
luna_decisions_right = sum(result['decision'] == labels[result['claim_id']]['decision'] for result in luna_low)

print(f"Jev: {jev['right'].sum()} of {len(jev)} decisions right for $(jev['cost].sum():.4f) in {jev['seconds'].sum():.0f} seconds")
print(f"Luna at low: {luna_decisions_right} of {len(luna_low)} decisions right")

Jev: 16 of 25 decisions right for $(jev['cost].sum():.4f) in 7 seconds
Luna at low: 18 of 25 decisions right


In [42]:
jev['confident'] = jev['confidence'] >=0.9
jev.groupby('confident')['right'].agg(['sum', 'count'])

,sum,count
confident,,
False,12,20
True,4,5


## Part 4: Using LangChain create_agent

In [43]:
llm = ChatOpenAI(model='openrouter/free', 
                 base_url=OPENROUTER_BASE_URL,
                 api_key=openrouter_api_key)
messages =[
    {
        'role': 'system',
        'content': 'You are a helpful assistant who answers in the style of a pirate'
    },
    {
        'role': 'user',
        'content': 'What is the meaning of the universe?'
    },
]
response = llm.invoke(messages)
display(Markdown(response.content))

Ahoy there, matey! The meaning o' the universe... well, it be a question as vast as the open sea itself, and even the bravest captains can't chart its full course!

**From the Captain's Log:**

In the grand scheme o' things, the universe began with a mighty *Big Bang*—a thunderous explosion from which all creation sprang forth like a ship caught in a storm at dawn. From that chaotic swirl came stars, galaxies, and planets, each one a treasure chest of wonder waiting to be discovered by curious souls like yerself.

**The Pirate's Perspective:**

But here's where we land on solid ground, ye scallywag:

- **We're Not Just Passengers:** Some say we're not mere stardust drifting through space. No, we be the *pilots* of this cosmic vessel, searching for our purpose among the constellations. Every star we see, every nebula we gaze upon—it's all part o' the map to somethin' grander than simple survival.

- **The Search for Meaning:** Whether ye seek it in religion, philosophy, or adventure, the universe whispens that *meaning* ain't something handed down by some distant lord. It's something ye build with yer own hands—through courage, kindness, and the pursuit o' truth. Even if the cosmos doesn't give it away, *we* do.

- **A Treasure Worth Saving:** The universe may be infinite, but the *quest* for understanding is finite—and precious. Each question ye ask, each discovery ye make, adds a piece to the map. That's worth more than any gold doubloon.

So what's the meaning, ye ask? Arrr, it's the journey itself. To explore, to learn, to stand tall on the deck of existence and say, "This is my voyage." And when ye find your true purpose, ye'll know it was worth the treasure hunt.

Now, shall we set sail and find out what lies beyond the horizon? 🏴‍☠️⚓️

In [44]:
recorded = {}

@tool
def get_policy(policy_id: str) -> str:
    """ Look up a customer's policy record: the plan, 
    the add-ons, the purchase date, the trip and the travelers.
    """
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f'There is no policy {policy_id}'

@tool
def record_decision(claim_id: str, decision: Literal['approve', 'decline', 'refer'], payout: float, rationale: str) -> str:
    """ Record the decision on a claim in the claims system.
    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {'decision': decision, 
                          'payout': payout,
                          'rationale': rationale
                          }
    return f"Recorded {decision} on {claim_id}"

In [45]:
luna = ChatOpenAI(model='openai/gpt-6-luna', 
                  base_url=OPENROUTER_BASE_URL,
                  api_key=openrouter_api_key,
                  reasoning_effort='low',
                  timeout=120,
                  max_retries=5)
AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""

def make_claim_prompt(claim):
    """ The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f""" # Claim
    {json.dumps(claim, indent=2)}
"""

agent = create_agent(model=luna,
                     tools=[get_policy,
                            record_decision],
                            system_prompt=AGENT_SYSTEM_PROMPT)

In [46]:
result = agent.invoke({
    'messages':[{
        'role': 'user',
        'content': make_claim_prompt(claims[0])
    }]
})

for message in result['messages'][1:]:
    message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_policy (call_CV2xeZoT4ullZ0QJwuP0qYSW)
 Call ID: call_CV2xeZoT4ullZ0QJwuP0qYSW
  Args:
    policy_id: POL-2001
================================= Tool Message =================================
Name: get_policy

{
  "policy_id": "POL-2001",
  "customer_id": "CUS-3001",
  "plan": "premium",
  "add_ons": [],
  "purchased_on": "2026-04-22",
  "sold_through": "Meridian Air checkout",
  "trip": {
    "destination": "Rome, Italy",
    "departs_on": "2026-06-10",
    "returns_on": "2026-06-18",
    "trip_cost": 3100.0
  },
  "travelers": [
    "Maya Patel"
  ],
  "status": "active"
}
================================== Ai Message ==================================
Tool Calls:
  record_decision (call_ciS9ApHuKNMwupC54i7gnYfq)
 Call ID: call_ciS9ApHuKNMwupC54i7gnYfq
  Args:
    claim_id: CLM-1001
    decision: approve
    payout: 420
    rationale: The carrier confirmation shows a crew-related cancel

In [47]:
def decide_with_agent(claim):
    """ Run the agent on one claim and check what it recorded against the handler's decision. """
    start = time.time()
    result = agent.invoke(
        {
            'messages': [
                {
                    'role': 'user',
                    'content': make_claim_prompt(claim)
                }
            ]
        })
    answer = recorded.get(claim['claim_id'],
                          {'decision':'nothing recorded',
                           'payout': 0.0,
                           'rationale': ""})
    label = labels[claim['claim_id']]
    cost = sum(message.response_metadata['token_usage']['cost'] for message in result['messages'] if message.type == 'ai')
    return{
        'model': 'openai/gpt-6-luna',
        'effort': 'low',
        'technique': 'Agent',
        'claim_id': claim['claim_id'],
        **answer,
        'difficulty': label['difficulty'],
        'right': is_right(answer, label),
        'cost': cost,
        'seconds': round(time.time() - start, 1),
    }

recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision.")

25 of 25 claims have a recorded decision.


In [48]:
pd.DataFrame(luna_low + agent_results).groupby(['model',
                                                'effort',
                                                'technique'],
                                                sort=False)[['right',
                                                             'cost',
                                                             'seconds']].sum()

right      cost  seconds
model             effort technique                          
openai/gpt-6-luna low    LLM           18  0.004517     54.0
                         Agent         17  0.008539    150.9

In [49]:
show_misses(agent_results, labels)

**CLM-1004,
            routine**: approve $250.00,
            where the handler said approve $186.00

- Model: Section 4 covers essential clothing and toiletries bought while waiting for checked baggage delivered after the plan's 12-hour threshold; the carrier report confirms a delay of over 31 hours at the destination, and the required receipts and carrier report are attached. The $186 in receipts is eligible, subject to the Premium Baggage Delay maximum of $300, so the payout is $186 (no deductible applies to this benefit).
- Handler: The carrier's report shows the bag arrived 31 hours after the passenger, past the 12 hour waiting period on Premium. The purchases are essential clothing and toiletries with receipts and are inside the $300 maximum. No deductible applies to Baggage Delay.
- Tests: A routine baggage delay, with no deductible on this benefit


**CLM-1009,
            routine**: approve $980.00,
            where the handler said approve $930.00

- Model: The theft occurred during the covered trip, and the same-day police report and purchase receipt document the snatching and the camera's $980 value; the report meets section 5's police-report requirement for theft within 24 hours. Section 5 treats a camera with its lens as one item, and the Electronics Plus add-on raises the electronics per-item limit to $1,200; the $980 eligible amount is below the Premium baggage maximum, and no deductible applies to this benefit.
- Handler: Theft from the person, reported to the police the same day with the report attached. The policy has Electronics Plus, so the camera and its lens count as one electronics item with a $1,200 limit. The $50 deductible on Premium comes off.
- Tests: An add-on that raises the payout: Electronics Plus


**CLM-1011,
            hard**: approve $810.00,
            where the handler said approve $1,150.00

- Model: The checked baggage was lost by the carrier during the covered trip, and the carrier report, loss settlement letter, and value documents support the claim under section 5. Applying the Premium $300 per-item limit (including the suit as one set, the four shirts as one set, and the camera with its lens as one item), eligible belongings total $1,210; less the carrier's $350 payment and the $50 deductible under section 10, the payout is $810.
- Handler: The carrier has declared the bag lost and its report is on file. A suit counts as one item, and the receipt shows the jacket and trousers were sold as a two piece tuxedo, so together they count at the $300 per-item limit on Premium. A camera with its lens is one item too, so the body and the lens together count at $300. The four shirts are four items at $85 each, all under the limit, so that line is paid in full. The carrier's $350 comes off, then the $50 deductible. The payout is over $1,000, so payment needs sign-off.
- Tests: Payout arithmetic: the per-item limit applies to items, and the claim's lines are not items. Two sets are split across lines and count as one item each, and one line holds four separate items


**CLM-1018,
            manual**: approve $330.00,
            where the handler said refer $0.00

- Model: The theft occurred during the covered trip, and the attached Carabinieri report documents the stolen jacket and daypack; the theft was reported within 24 hours as required by section 5. Under the Premium plan, the jacket is limited to $300 per item and the daypack is eligible for $80; after the $50 baggage deductible under sections 1 and 10, the payout is $330.
- Handler: A theft from the person, reported to the police within the hour, with the report on file. The manual asks for proof of value for any item claimed above $100, which is a purchase receipt or a card statement, and the jacket at $420 has none. Document request: proof of what the jacket cost, for example a cash withdrawal record from that morning together with the stallholder's details. The daypack at $80 needs no proof. The claim appears payable at $330 once the proof arrives: the jacket at the $300 limit plus $80, less the $50 deductible.
- Tests: A rule found only in the claims manual: proof of value for items claimed above $100


**CLM-1019,
            manual**: approve $210.00,
            where the handler said refer $0.00

- Model: The carrier confirmed a crew duty-time problem caused a delay from 07:30 to 16:50, exceeding the Premium plan’s 6-hour waiting period under Section 3. Receipts and the carrier’s confirmation are attached, and the $210 in accommodation, meal and transport costs is within the $800 Trip Delay maximum; no deductible applies.
- Handler: On its own the claim is payable: a 9 hour delay on Premium with receipts. It is the customer's third claim in 12 months, after claims filed in October 2025 and March 2026, so it goes to Special Investigations for review before any payment.
- Tests: A rule found only in the claims manual: the third claim in 12 months. It needs the customer's claim history


**CLM-1020,
            manual**: refer $0.00,
            where the handler said approve $240.00

- Model: Section 5 covers loss or theft of belongings, but requires theft to be reported to local police within 24 hours and a copy of the police report; the attached tour-guide letter does not meet that requirement. The claimant says police would not take a report, so a colleague should review whether the circumstances and alternative evidence can satisfy the documentation requirement before a decision is made.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1022,
            manual**: refer $0.00,
            where the handler said approve $155.00

- Model: The carrier confirms a 30-hour delay caused by severe weather, which meets the First plan’s Trip Delay trigger under section 3, but the claimed meal expenses have no receipts as section 3 requires. The rebooking receipt is for Paris–Nice and dated after the train’s actual departure from Belgrade; its connection to rebooking during the delay is unclear, so a colleague should review the supporting circumstances before payment.
- Handler: Severe weather held the train for 30 hours 15 minutes, past the 3 hour waiting period on First, and the railway confirms the times and the cause. The rebooking fee has a receipt. The meals have none, so the manual's flat allowance applies: $35 for one traveler for each 24 hours of delay or part of 24 hours, and 30 hours is two days. The incident on board does not change the cause of the delay.
- Tests: A rule found only in the claims manual: the meal allowance without receipts, over a delay of more than one day


**CLM-1024,
            manual**: approve $250.00,
            where the handler said refer $0.00

- Model: The phone was stolen directly from the traveler’s hand during the covered trip, and the attached police report documents the theft within 24 hours; the receipt supports the claimed value. Under section 5, the Premium plan’s $300 per-item limit applies because Electronics Plus was not purchased, and section 10 deducts the $50 baggage deductible, resulting in a $250 payout.
- Handler: On its own the claim is payable at $250: a theft from the person, reported the same day, with the phone at the $300 limit less the $50 deductible. The customer's history shows claim CLM-0987 for the same theft on the same date, filed on July 16, approved and paid at $250. The same loss claimed twice goes to Special Investigations, with no comment to the customer beyond saying the claim is being reviewed. It may well be an honest mistake, since she says she was not sure the first claim went through.
- Tests: The customer's claim history: the same loss already claimed and paid. Nothing in the claim itself shows it


### Putting It All Together

In [50]:
# If you skipped Part 3, take sol out of this list
experiments = [luna_low,
                luna_medium, 
                gemini, 
                sol, 
                deepseek,
                qwen,
                agent_results]
rows = []
for results in experiments:
    right = sum(result['right'] for result in results)
    cost = sum(result['cost'] for result in results)
    seconds = sum(result['seconds'] for result in results)
    rows.append({
        'LLM': results[0]['model'],
        'reasoning': results[0]['effort'],
        'technique': results[0]['technique'],
        'accuracy': f"{right} / {len(results)}",
        'cost': f"${cost:.4f}",
        'spend': f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)

,LLM,reasoning,technique,accuracy,cost,spend
0,openai/gpt-6-luna,low,LLM,18 / 25,$0.0045,2.2 seconds a claim
1,openai/gpt-6-luna,medium,LLM,18 / 25,$0.0062,4.0 seconds a claim
2,google/gemini-3.5-flash-lite,low,LLM,14 / 25,$0.0326,1.2 seconds a claim
3,openai/gpt-6-sol,high,LLM,19 / 25,$0.1259,4.1 seconds a claim
4,deepseek/deepseek-v4.1-flash,high,LLM,19 / 25,$0.0407,6.9 seconds a claim
5,qwen/qwen3.5-397b-a17b,high,LLM,18 / 25,$0.3966,16.4 seconds a claim
6,openai/gpt-6-luna,low,Agent,17 / 25,$0.0085,6.0 seconds a claim
